# Inverse B with the energy spectrum (internal comparison)

The model of `inverse_b_pipeline.ipynb` with the first 4 principal components of the energy spectrum (`spec_common` / its total) as extra inputs. Not a submission: the test file has no spectrum. Same split, so the scores compare directly. The spectrum comes from the pack's `campaign_B.parquet`.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import shutil

import numpy as np
import pandas as pd

import pallas_pipeline as pp
import pallas_score

project = pp.connect(pp.PROJECT)
fs = project.get_feature_store()
case = pp.CASES["inverse_B"]
module = pp.load_model_file("models/inverse_b_regress_mdn.py")
seed, register = 12, False


def mixture_scores(est, X, rows, n_draw=2000):
    """NLL (standardised units), 90% interval coverage and width of the mixture."""
    w, mu, sd = est.predict_mixture(np.asarray(X, float))
    true = rows[case.targets].to_numpy(float)
    log_n = (-0.5 * ((true[:, None] - mu) / sd) ** 2 - np.log(sd / est.y_scaler.scale_) - 0.5 * np.log(2 * np.pi)).sum(2)
    top = log_n.max(1, keepdims=True)
    rng = np.random.default_rng(0)
    pick = np.minimum((rng.random((len(w), n_draw, 1)) > w.cumsum(1)[:, None, :]).sum(2), w.shape[1] - 1)
    r = np.arange(len(w))[:, None]
    draws = mu[r, pick] + sd[r, pick] * rng.standard_normal((len(w), n_draw, len(case.targets)))
    low, high = np.quantile(draws, 0.05, 1), np.quantile(draws, 0.95, 1)
    return {"nll": -(top[:, 0] + np.log((w * np.exp(log_n - top)).sum(1))).mean(),
            **{f"cover90_{pp.fs_name(t)}": c for t, c in zip(case.targets, ((true >= low) & (true <= high)).mean(0))},
            **{f"width90_{pp.fs_name(t)}": v for t, v in zip(case.targets, (high - low).mean(0))}}

/home/mlenivenko/miniconda3/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-15 16:23:45,244 INFO: Initializing external client


2026-10-15 16:23:45,245 INFO: Base URL: https://aissai26.hops.live:443


2026-10-15 16:23:46,500 INFO: Python Engine initialized.



Logged in to project, explore it here https://aissai26.hops.live:443/p/2170


In [2]:
fv = pp.get_feature_view(fs, "m_inverse_b_raw_fv", 1)
td_version = pp.prepare_statistics(fv)
spectra = pd.read_parquet(pp.DATA_DIR / "campaign_B.parquet", columns=["config", "spec_common"])
spec = [f"spec_{i:03d}" for i in range(200)]
rows = pp.read_rows(fv, case).merge(pd.concat([spectra[["config"]], pd.DataFrame(np.stack(spectra.spec_common), columns=spec)], axis=1),
                                    on="config")
train_configs, holdout_configs, _ = pp.split_configs(rows, case, "random", 0.2, seed)
train = rows[rows.config.isin(train_configs)].reset_index(drop=True)
holdout = rows[rows.config.isin(holdout_configs)].reset_index(drop=True)
X_of = lambda r: pd.concat([pp.transform(fv, r, case), r[spec]], axis=1)
X_train, X_holdout = X_of(train), X_of(holdout)
print(f"{len(train)} train, {len(holdout)} holdout configs, {X_train.shape[1]} input columns")

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (0.89s) 
2306 train, 576 holdout configs, 209 input columns


In [3]:
params = {**module.PARAMS, "columns": list(X_train.columns), "n_spectrum_pca": 4}
model = pp.SurrogateModel(module.build_model(**params), case.targets, "scorer").fit(X_train, train)
print(f"spectrum variance in 4 components: {model.estimator.pca.explained_variance_ratio_.sum():.3f}")
scores = pallas_score.score_submission(pp.predictions(case, model, X_holdout, holdout), pp.answer_key(case, holdout))
metrics = {"holdout_mean_r2": scores["mean"], **{f"holdout_r2_{pp.fs_name(t)}": scores[t] for t in case.targets},
           **{f"holdout_{k}": v for k, v in mixture_scores(model.estimator, X_holdout, holdout).items()}}
if register:
    info = {"case": "inverse_B", "inputs": case.inputs + spec, "targets": case.targets, "feature_columns": list(X_train.columns),
            "feature_view": fv.name, "feature_view_version": fv.version, "td_version": td_version,
            "model_file": "inverse_b_regress_mdn", "params": params, "target_scaling": "scorer", "seed": seed}
    bundle = pp.save_bundle(model, "models/inverse_b_regress_mdn.py", info, Path("team_models/m_inverse_b_regress_mdn_spectrum_pca4"))
    pp.register_model(project, bundle, "m_inverse_b_regress_mdn_spectrum_pca4", fv, td_version, metrics,
                      "inverse_B internal comparison, not for submission: + 4 spectrum principal components")
    shutil.rmtree(bundle)
pd.Series(metrics).round(4)

spectrum variance in 4 components: 0.698


holdout_mean_r2               0.9225
holdout_r2_p_max              0.9789
holdout_r2_cn2_max            0.9673
holdout_r2_l_inj              0.9599
holdout_r2_dip_frac           0.7837
holdout_nll                  -1.8902
holdout_cover90_p_max         0.9219
holdout_cover90_cn2_max       0.9375
holdout_cover90_l_inj         0.9323
holdout_cover90_dip_frac      0.9479
holdout_width90_p_max       389.4026
holdout_width90_cn2_max       0.0352
holdout_width90_l_inj         0.0769
holdout_width90_dip_frac      0.2113
dtype: float64